# Unified Image Studio — MOSS + Anima (Tailscale Funnel)
One ComfyUI server for both families, designed for Colab Free (T4 GPU, ~15 GiB usable VRAM and ~12.7 GiB system RAM). Choose **one** model, download only its components, and generate inside the notebook. No browser UI required.

**Quick start:** choose GPU → run sections 1–5 (section 3b starts the supervisor) → optionally upload reference or LoRAs → set generation controls → generate. **Reconnect:** reattach the runtime, mount Drive in section 2, run 3b again, then 3–5 and the tunnel; the supervisor restarts a dead ComfyUI or tunnel by itself while the runtime lives. For switching models, pause any remote clients, run section 3 again with the new choice, then section 4 and the LoRA panel/configuration cells. A switch unloads the previous process; files remain cached on disk. No full runtime reset is required. Never switch while a job is running.

Anima Turbo v1.1 is the lightweight default; MOSS uses the same INT8 + W4A8 checkpoint configuration as the previously working MOSS notebook. Reference Adapter is Anima-only; Base v1.0 is the validated target. Model weights and LoRAs have their own licenses.

Authentication is optional: the tunnel cell is opt-in, and its `TUNNEL_MODE` selector starts either a Tailscale Funnel (stable URL) or a cloudflared quick tunnel (new URL every session). Google restricts remote proxy connections on managed Colab and certain external-UI uses on Free; use the tunnel only on a host that permits it. This notebook does not provision GPU runtimes, defeat idle limits, or implement the separate Spindle extension.

Sources: [Anima model card](https://huggingface.co/circlestone-labs/Anima), [MOSS ComfyUI weights](https://huggingface.co/Comfy-Org/MOSS-Image), [Anima Reference Adapter](https://github.com/ArcAnima/ComfyUI-Anima-Ref-Adapter), [Colab FAQ](https://research.google.com/colaboratory/faq.html), [cloudflared](https://github.com/cloudflare/cloudflared).

Credentials come from three places, in order: the environment, `MyDrive/rgm-tailnet/credentials.json`, then Colab secrets. None of them is this notebook — it is public.


## 1 · GPU and workspace
Use **Runtime → Change runtime type → T4 GPU**. Not CPU or TPU. This check avoids importing PyTorch into the notebook kernel to conserve RAM.

In [ ]:
import os, sys, time, subprocess, shutil, json, random, io, gc
from pathlib import Path
# Set by the automated bootstrap (RGM_AUTOMATION=1). When true, nothing may
# block waiting for a browser: no file picker, no getpass, no input().
AUTOMATION = os.environ.get('RGM_AUTOMATION', '').strip().lower() in ('1', 'true', 'yes')
import requests, psutil
COMFY = Path('/content/Unified_ComfyUI')
MODELS = COMFY / 'models'
BASE_URL = 'http://127.0.0.1:8188'
LOG = Path('/content/unified_comfyui.log')
ADAPTER = COMFY / 'custom_nodes' / 'ComfyUI-Anima-Ref-Adapter'
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True).strip())
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError('Select an NVIDIA GPU runtime (T4 or better).') from e

def resources():
    ram=psutil.virtual_memory()
    print(f'System available RAM: {ram.available/1024**3:.2f} GiB / {ram.total/1024**3:.2f} GiB total')
    subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader'])
    print(f'Local disk free: {shutil.disk_usage("/content").free/1024**3:.1f} GiB')
resources()

## 2 · Install ComfyUI and configure storage
Drive caching is optional and off by default. MOSS alone requires about 13 GB of weight files; caching multiple checkpoints can exceed a free Drive allocation. Anima and MOSS share one server but not model components. Leave updates off after a working install unless needed.

In [ ]:
# @title Installation and storage
# Drive keeps weights, cache and the tailnet identity across runtimes. Turn it
# off only if you accept a full re-download on every reconnect.
USE_DRIVE = True # @param {type:"boolean"}
UPDATE_COMFYUI = False # @param {type:"boolean"}
INSTALL_ANIMA_ADAPTER = True # @param {type:"boolean"}
CACHE = Path('/content/unified_model_cache')
# Headless run (RGM_AUTOMATION=1): there is no browser to complete the Drive
# consent, so mount would block forever. Fall back to /content for this run.
if globals().get('AUTOMATION') and USE_DRIVE:
    print('Automation: Drive mount skipped (no browser); using /content.')
    USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE=Path('/content/drive/MyDrive/Unified_Image_Assets')
CACHE.mkdir(parents=True,exist_ok=True)
# Credentials never go in this notebook: it is committed to a public repo.
# Yours live on Drive instead, which is private to your account.
#   MyDrive/rgm-tailnet/credentials.json
#   {"BASIC_AUTH_USER": "...", "BASIC_AUTH_PASS": "...", "TS_AUTHKEY": "tskey-auth-..."}
# Environment variables win, so the Space or the automation can override the file.
CREDENTIALS_FILE = Path('/content/drive/MyDrive/rgm-tailnet/credentials.json')
if USE_DRIVE and CREDENTIALS_FILE.is_file():
    try:
        _credentials = json.loads(CREDENTIALS_FILE.read_text(encoding='utf-8'))
        for _name, _value in _credentials.items():
            if str(_value).strip():
                os.environ.setdefault(str(_name), str(_value))
        print(f'Credentials: loaded {len(_credentials)} keys from {CREDENTIALS_FILE}')
    except Exception as _exc:
        print('Credentials: could not read', CREDENTIALS_FILE, '->', _exc)
elif USE_DRIVE:
    print('Credentials: none at', CREDENTIALS_FILE, '(Colab secrets still work)')
# Use /content for HF transient cache even when Drive persistence is enabled.
os.environ.setdefault('HF_HOME','/content/unified_hf_cache')
def run_cmd(*args,cwd=None):
    subprocess.run([str(x) for x in args],cwd=cwd,check=True)
if not (COMFY/'main.py').is_file():
    run_cmd('git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',COMFY)
elif UPDATE_COMFYUI:
    run_cmd('git','pull','--ff-only',cwd=COMFY)
for category in ('diffusion_models','text_encoders','vae','loras'):
    (MODELS/category).mkdir(parents=True,exist_ok=True)
run_cmd(sys.executable,'-m','pip','install','-q','-r',COMFY/'requirements.txt')
run_cmd(sys.executable,'-m','pip','install','-q','huggingface_hub>=0.25','requests','pillow','ipywidgets','safetensors','psutil')
if INSTALL_ANIMA_ADAPTER and not ADAPTER.exists():
    run_cmd('git','clone','--depth','1','https://github.com/ArcAnima/ComfyUI-Anima-Ref-Adapter.git',ADAPTER)
print('Cache:',CACHE)
print('ComfyUI revision:', subprocess.check_output(['git','rev-parse','--short','HEAD'],cwd=COMFY,text=True).strip())

## 3 · Select and prepare a model
Only the selected checkpoint, encoder, and VAE are installed. Anima variants reuse their encoder/VAE. Model changes stop the owned ComfyUI server after checking its queue; **pause remote clients before switching**. Rerun section 4 to start the new session, then sections 5–9 as applicable. Existing weights stay on disk, not loaded in RAM. Do not edit the model name in the generation cell.

In [ ]:
# @title Model selection (rerun to switch)
MODEL_CHOICE = 'MOSS SFT INT8' # @param ['Anima Turbo v1.1','Anima Aesthetic v1.1','Anima Base v1.0','Anima Turbo v1.0','Anima Aesthetic v1.0','Anima Aesthetic v1.0b','MOSS SFT INT8','MOSS Base INT8','MOSS SFT BF16','MOSS Base BF16','Anima Preview','Anima Preview 2','Anima Preview 3 Base','Anima 2.9B Preview BF16','Anima 2.9B Preview INT8']
DOWNLOAD_ADAPTER_WEIGHTS = False # @param {type:"boolean"}
from huggingface_hub import hf_hub_download
from safetensors import safe_open

ANIMA_REPO='circlestone-labs/Anima'
MOSS_REPO='Comfy-Org/MOSS-Image'
def make_profile(choice):
    if choice.startswith('MOSS '):
        flavor='sft' if 'SFT' in choice else 'base'
        return dict(family='moss',repo=MOSS_REPO,root='',checkpoint=f'moss_image_{flavor}_{"bf16" if "BF16" in choice else "int8_convrot"}.safetensors',encoder='qwen3.5_9b_w4a8_gptq.safetensors',vae='flux2-vae_small.safetensors',clip_type='moss_image',latent='EmptyFlux2LatentImage',steps=25,cfg=2.5,sampler='er_sde',scheduler='simple')
    names={'Anima Preview':'anima-preview.safetensors','Anima Preview 2':'anima-preview2.safetensors','Anima Preview 3 Base':'anima-preview3-base.safetensors','Anima 2.9B Preview BF16':'Anima-2.9B-preview-v1.safetensors','Anima 2.9B Preview INT8':'Anima-2.9B-preview-v1_int8_convrot.safetensors','Anima Base v1.0':'anima-base-v1.0.safetensors','Anima Aesthetic v1.0':'anima-aesthetic-v1.0.safetensors','Anima Aesthetic v1.0b':'anima-aesthetic-v1.0b.safetensors','Anima Aesthetic v1.1':'anima-aesthetic-v1.1.safetensors','Anima Turbo v1.0':'anima-turbo-v1.0.safetensors','Anima Turbo v1.1':'anima-turbo-v1.1.safetensors'}
    turbo='Turbo' in choice
    return dict(family='anima',repo=ANIMA_REPO,root='split_files/',checkpoint=names[choice],encoder='qwen_3_06b_base.safetensors',vae='qwen_image_vae.safetensors',clip_type='stable_diffusion',latent='EmptyLatentImage',steps=10 if turbo else 30,cfg=1.0 if turbo else 4.5,sampler='er_sde',scheduler='simple')

def comfy_ready():
    try: return requests.get(BASE_URL+'/system_stats',timeout=3).ok
    except requests.RequestException: return False

def require_idle():
    r=requests.get(BASE_URL+'/queue',timeout=10);r.raise_for_status();q=r.json()
    if q.get('queue_running') or q.get('queue_pending'):
        raise RuntimeError('ComfyUI has active or queued jobs. Wait until finished before changing models or stopping.')

def stop_owned_server():
    proc=globals().get('comfy_proc')
    if proc is None or proc.poll() is not None:
        if comfy_ready(): raise RuntimeError('Port 8188 is occupied by an unmanaged ComfyUI process. Stop the old notebook server first.')
        return
    if comfy_ready(): require_idle()
    proc.terminate()
    try: proc.wait(timeout=30)
    except subprocess.TimeoutExpired: proc.kill();proc.wait(timeout=15)
    print('Previous ComfyUI stopped; cached model RAM/VRAM released.')

p=make_profile(MODEL_CHOICE)
if comfy_ready() and (globals().get('RUNNING_CHOICE')!=MODEL_CHOICE or globals().get('ACTIVE_PROFILE')!=p):
    stop_owned_server()
ACTIVE_PROFILE=None  # Set only when all requested assets are ready.
LORA_CONFIG=[]; LORA_VALIDATED_FAMILY=None; LORA_VALIDATED_SIGNATURE=None
LORA_DIR=MODELS/'loras'/p['family'];LORA_DIR.mkdir(parents=True,exist_ok=True)
LORA_CACHE=CACHE/'loras'/p['family'];LORA_CACHE.mkdir(parents=True,exist_ok=True)

def acquire_weight(repo, repo_path, category):
    name=Path(repo_path).name
    dest=MODELS/category/name
    if dest.is_file() and dest.stat().st_size>100000:
        print('Installed:',name);return dest
    # local_dir avoids the extra full Hugging Face snapshot copy.
    storage=CACHE/'weights'/repo.replace('/','--')
    cached=Path(hf_hub_download(repo_id=repo,filename=repo_path,local_dir=storage))
    if dest.exists():dest.unlink()
    try: os.link(cached,dest)
    except OSError:
        if shutil.disk_usage(dest.parent).free < cached.stat().st_size + 512*1024**2:
            raise OSError('Not enough local disk space to stage model weights.')
        shutil.copy2(cached,dest)
    print('Ready:',name,f'({dest.stat().st_size/1024**3:.2f} GiB)')
    return dest

for category,key in [('diffusion_models','checkpoint'),('text_encoders','encoder'),('vae','vae')]:
    is_29b = category == 'diffusion_models' and MODEL_CHOICE.startswith('Anima 2.9B')
    acquire_weight('Gazingstars123/Anima-2.9B' if is_29b else p['repo'], p[key] if is_29b else p['root']+category+'/'+p[key], category)
if p['family']=='anima' and DOWNLOAD_ADAPTER_WEIGHTS:
    acquire_weight('ElectronGirl/anima-ref-adapter','anima_ref_adapter_v2_r64.safetensors','loras')
# Restore only this family's LoRAs. The adapter has a separate root filename.
for src in LORA_CACHE.glob('*.safetensors'):
    dst=LORA_DIR/src.name
    if USE_DRIVE and not dst.exists():
        shutil.copy2(src,dst)
ACTIVE_PROFILE=p
print('Selected:',MODEL_CHOICE,'| default:',p['steps'],'steps, CFG',p['cfg'])
resources()

## 3b · Worker lifecycle: supervisor and keep-alive
Section 3 prepared the model; this starts the **supervisor**, a background thread that looks after the worker for the rest of the session. Every `BEAT_SECONDS` it checks ComfyUI and the tunnel, restarts children that have exited, probes the public URL through the relay every `PROBE_EVERY_BEATS` beats, and mirrors the Tailscale state to Drive every `MIRROR_EVERY_BEATS`. It writes `/content/worker-status.json` and a copy under `MyDrive/rgm-tailnet/`, so the next runtime can report what the previous one did.

The beat is real work — probe, restart, mirror — not a simulated click. It does **not** extend Colab's idle or 12-hour limits and is not meant to. What it buys you is a worker that recovers from a dead child process instead of failing silently, and a record of when and why it stopped.

`RESTART_HUNG_COMFY` is off by default: a ComfyUI that is alive but slow is more often loading a model than dead, and killing it would destroy a job. Tick it only if you see repeated "alive but not answering" warnings with no progress. Tunnel restarts are capped at three per hour, because a daemon that dies four times in an hour is broken, not unlucky.

In [ ]:
# @title Worker lifecycle: supervisor, keep-alive, session report
LIFECYCLE = True            # @param {type:"boolean"}
BEAT_SECONDS = 30           # @param {type:"integer"}
PROBE_EVERY_BEATS = 10      # @param {type:"integer"}
MIRROR_EVERY_BEATS = 30     # @param {type:"integer"}
RESTART_HUNG_COMFY = False  # @param {type:"boolean"}

import threading, uuid, datetime

STATUS_PATH = Path('/content/worker-status.json')
TS_RUN = Path('/content/ts-run')
TS_PERSIST = Path('/content/drive/MyDrive/rgm-tailnet')

# One dict, not module state, so a cell rerun in the same kernel keeps its identity
# and the report can tell "previous runtime" from "me, one cell ago".
worker = globals().get('worker')
if not isinstance(worker, dict):
    worker = {}
worker.setdefault('kernel', uuid.uuid4().hex[:8])
worker.setdefault('started_at', time.time())
worker.setdefault('beats', 0)
worker.setdefault('restarts', 0)
worker.setdefault('last_beat', None)
worker.setdefault('last_probe', None)
worker.setdefault('last_error', None)
worker.setdefault('events', [])
worker.setdefault('errors', {})
worker.setdefault('comfy_armed', False)

def _set_error(key, message=None):
    """Track failures per subsystem so a good restart clears only its own error."""
    errors = worker.setdefault('errors', {})
    if message is None:
        errors.pop(key, None)
        worker['last_error'] = next(iter(errors.values()), None)
    else:
        errors[key] = message
        worker['last_error'] = message

def wlog(msg, level='info'):
    print(f"[{datetime.datetime.now():%H:%M:%S}] worker: {msg}")
    worker['events'].append({'t': time.time(), 'level': level, 'msg': msg})
    del worker['events'][:-200]

def _stamp(when):
    return datetime.datetime.fromtimestamp(when).strftime('%m-%d %H:%M') if when else 'never'

def worker_report():
    """Say what the previous runtime did before this one overwrites the evidence."""
    if not STATUS_PATH.is_file():
        print('worker: no previous status file; this is a fresh runtime.')
        return
    try:
        prev = json.loads(STATUS_PATH.read_text())
    except Exception:
        return
    if prev.get('kernel') == worker['kernel']:
        return
    begin, end = prev.get('started_at'), prev.get('last_beat')
    mins = int(((end or begin or 0) - (begin or 0)) / 60)
    print(f"worker: previous runtime {_stamp(begin)} -> {_stamp(end)} "
          f"({mins} min, {prev.get('beats', 0)} beats, {prev.get('restarts', 0)} restarts)")
    if prev.get('model'):
        print('worker:   model', prev['model'])
    if prev.get('url'):
        print('worker:   tunnel', prev['url'])
    if prev.get('last_error'):
        print('worker:   last error', prev['last_error'])

def _save_status(comfy_ok=None):
    data = {
        'kernel': worker['kernel'],
        'started_at': worker['started_at'],
        'last_beat': worker.get('last_beat'),
        'beats': worker['beats'],
        'restarts': worker['restarts'],
        'last_error': worker.get('last_error'),
        'model': globals().get('MODEL_CHOICE'),
        'vram': globals().get('RUNNING_VRAM') or globals().get('VRAM_MODE'),
        'comfy': comfy_ok,
        'url': globals().get('URL') or globals().get('TUNNEL_URL'),
        'probe': worker.get('last_probe'),
    }
    for path in (STATUS_PATH, TS_PERSIST.parent / 'worker-status.json'):
        try:
            if path.parent.is_dir():
                tmp = path.with_suffix('.tmp')
                tmp.write_text(json.dumps(data, indent=1))
                os.replace(tmp, path)
        except Exception:
            pass

def start_comfy(vram_mode=None):
    """Start ComfyUI if it is not answering. Returns True when it answers."""
    global comfy_proc, RUNNING_CHOICE, RUNNING_VRAM
    if comfy_ready():
        worker['comfy_armed'] = True
        return True
    mode = (vram_mode or globals().get('RUNNING_VRAM')
            or globals().get('VRAM_MODE') or 'normalvram')

    old = globals().get('comfy_proc')
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=30)
        except subprocess.TimeoutExpired:
            old.kill()
            old.wait(timeout=15)

    launch_args = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188']
    if mode == 'lowvram':
        launch_args.append('--lowvram')
    print('Launching ComfyUI:', ' '.join(launch_args))
    with LOG.open('a', buffering=1) as log_handle:
        comfy_proc = subprocess.Popen(
            launch_args, cwd=COMFY, stdout=log_handle, stderr=subprocess.STDOUT,
        )

    for _ in range(150):
        if comfy_ready() or comfy_proc.poll() is not None:
            break
        time.sleep(2)

    if not comfy_ready():
        print(LOG.read_text(errors='replace')[-7000:] if LOG.exists() else 'No log')
        return False
    RUNNING_VRAM = mode
    RUNNING_CHOICE = globals().get('MODEL_CHOICE')
    worker['comfy_armed'] = True
    return True

_supervisor_stop = threading.Event()
_supervisor_thread = None
_comfy_misses = 0

def _alive(name):
    proc = globals().get(name)
    return proc is not None and proc.poll() is None

def _check_comfy():
    global _comfy_misses
    if comfy_ready():
        _comfy_misses = 0
        return True
    if not worker.get('comfy_armed'):
        return False                      # nothing was started here; not ours to recover
    _comfy_misses += 1
    if not _alive('comfy_proc'):
        wlog('ComfyUI process exited; restarting')
        if start_comfy():
            worker['restarts'] += 1
            _set_error('comfy')
            wlog('ComfyUI restarted')
            return True
        _set_error('comfy', 'comfy_restart_failed')
        wlog(f'ComfyUI restart failed; see {LOG}', 'error')
        return False
    if _comfy_misses == 1:
        return False                      # one slow answer is not a death
    if _comfy_misses >= 3 and RESTART_HUNG_COMFY:
        wlog('ComfyUI is alive but unresponsive; restarting (RESTART_HUNG_COMFY)')
        if start_comfy():
            worker['restarts'] += 1
            _set_error('comfy')
            wlog('ComfyUI restarted')
            _comfy_misses = 0
            return True
    elif _comfy_misses == 3:
        wlog('ComfyUI is alive but not answering /system_stats; leaving it alone '
             '(tick RESTART_HUNG_COMFY to force a restart)', 'warn')
    return False

def _check_tunnel():
    mode = globals().get('TUNNEL_MODE')
    if mode not in ('tailscale', 'cloudflare'):
        return None
    down = []
    if not _alive('tunnel_caddy_proc'):
        down.append('caddy')
    if mode == 'tailscale' and not _alive('ts_proc'):
        down.append('tailscaled')
    if mode == 'cloudflare' and not _alive('tunnel_cf_proc'):
        down.append('cloudflared')
    if not down:
        return True
    restart = globals().get('restart_tunnel')
    if callable(restart):
        wlog('tunnel down (' + ', '.join(down) + '); restarting')
        try:
            restart()
            worker['restarts'] += 1
            _set_error('tunnel')
            wlog('tunnel restarted')
            return True
        except Exception as exc:
            _set_error('tunnel', f'tunnel_restart_failed: {exc}')
            wlog(f'tunnel restart failed: {str(exc)[:200]}', 'error')
            return False
    if worker.get('tunnel_warned') != down:
        worker['tunnel_warned'] = down
        wlog('tunnel down (' + ', '.join(down) + '); rerun the tunnel cell', 'warn')
    return False

def probe_public():
    """Health-check the worker the way a remote client sees it, through the relay."""
    url = globals().get('URL') or globals().get('TUNNEL_URL')
    if not url:
        return None
    cmd = ['curl', '-sS', '-o', '/dev/null', '-w', '%{http_code} %{time_total}',
           '--max-time', '25']
    if (TS_RUN / 'tailscaled.sock').exists():
        cmd += ['--proxy', 'socks5h://127.0.0.1:1055']
    user, password = globals().get('BASIC_USER'), globals().get('BASIC_PASS')
    if user and password:
        cmd += ['-u', f'{user}:{password}']
    cmd.append(url.rstrip('/') + '/system_stats')
    r = subprocess.run(cmd, text=True, capture_output=True)
    parts = (r.stdout or '').split()
    if len(parts) == 2 and parts[0].startswith('2'):
        worker['last_probe'] = {'ok': True, 'at': time.time(),
                                'ms': round(float(parts[1]) * 1000)}
    else:
        worker['last_probe'] = {'ok': False, 'at': time.time(),
                                'detail': (r.stderr or r.stdout).strip()[:200]}
    return worker['last_probe']

_restart_attempts = {}

def _allow_restart(key, limit=3, window=3600):
    now = time.time()
    hits = [t for t in _restart_attempts.get(key, []) if now - t < window]
    _restart_attempts[key] = hits
    if len(hits) >= limit:
        return False
    hits.append(now)
    return True

def restart_tunnel():
    """Bring Caddy and tailscaled back with the same URL. Quick tunnels excluded."""
    global tunnel_caddy_proc, ts_proc
    if globals().get('TUNNEL_MODE') != 'tailscale':
        raise RuntimeError('a quick tunnel gets a new hostname on restart; '
                           'rerun the tunnel cell and update the Space secret')
    if not _allow_restart('tunnel'):
        raise RuntimeError('3 tunnel restarts in an hour; giving up until the cell is rerun')
    caddy = Path('/content/caddy')
    launch = globals().get('TS_LAUNCH')
    ts = Path('/content/tailscale')
    sock = TS_RUN / 'tailscaled.sock'
    if not caddy.is_file() or not launch or not ts.is_file():
        raise RuntimeError('tunnel binaries or launch flags missing; rerun the tunnel cell')

    for name in ('tunnel_caddy_proc', 'ts_proc'):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except Exception:
                proc.kill()
    try:
        sock.unlink()
    except OSError:
        pass

    tunnel_caddy_proc = subprocess.Popen(
        [str(caddy), 'run', '--config', '/content/Caddyfile'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    TS_RUN.mkdir(parents=True, exist_ok=True)
    _log = open('/content/tailscaled.log', 'a')
    ts_proc = subprocess.Popen(launch, stdout=_log, stderr=subprocess.STDOUT)

    for _ in range(120):                      # daemon control socket
        if sock.exists():
            break
        if ts_proc.poll() is not None:
            raise RuntimeError('tailscaled exited again; see /content/tailscaled.log')
        time.sleep(0.5)
    else:
        raise RuntimeError('tailscaled never recreated its socket')

    for _ in range(120):                      # back to Running with the restored state
        r = subprocess.run([str(ts), '--socket', str(sock), 'status', '--json'],
                           text=True, capture_output=True)
        try:
            state = json.loads(r.stdout or '{}').get('BackendState')
        except Exception:
            state = None
        if state == 'Running':
            break
        time.sleep(1)
    else:
        raise RuntimeError(f'tailscaled did not return to Running (state={state})')

    subprocess.run([str(ts), '--socket', str(sock), 'funnel', '--bg',
                    '--https=443', '--yes', 'http://127.0.0.1:8189'],
                   text=True, capture_output=True)
    time.sleep(2)
    if not comfy_ready():
        raise RuntimeError('ComfyUI did not answer after the tunnel restart')
    print('tunnel: back up on the same URL')

def _mirror_state():
    if not TS_RUN.is_dir() or not TS_PERSIST.parent.is_dir():
        return
    src = TS_RUN / 'tailscaled.state'
    if not (src.is_file() and src.stat().st_size > 0):
        return
    try:
        tmp = TS_PERSIST / 'tailscaled.state.tmp'
        shutil.copy2(src, tmp)
        os.replace(tmp, TS_PERSIST / 'tailscaled.state')
        if (TS_RUN / 'certs').is_dir():
            shutil.copytree(TS_RUN / 'certs', TS_PERSIST / 'certs', dirs_exist_ok=True)
    except Exception as exc:
        wlog(f'state mirror failed: {str(exc)[:200]}', 'warn')

def supervisor_tick():
    worker['beats'] += 1
    beat = worker['beats']
    comfy_ok = False
    try:
        comfy_ok = bool(_check_comfy())
        _check_tunnel()
        if PROBE_EVERY_BEATS and beat % PROBE_EVERY_BEATS == 0:
            probe = probe_public()
            if probe and not probe.get('ok'):
                wlog('public probe failed: ' + str(probe.get('detail'))[:160], 'warn')
        if MIRROR_EVERY_BEATS and beat % MIRROR_EVERY_BEATS == 0:
            _mirror_state()
    except Exception as exc:
        _set_error('supervisor', f'{type(exc).__name__}: {exc}')
        wlog(f'supervisor error: {worker["last_error"]}', 'error')
    worker['last_beat'] = time.time()
    _save_status(comfy_ok)

def supervisor_start():
    global _supervisor_thread
    if _supervisor_thread is not None and _supervisor_thread.is_alive():
        print('worker: supervisor already running')
        return
    _supervisor_stop.clear()

    def loop():
        # A genuine beat: probe, restart what died, mirror state. This is not a
        # simulated click, and it does not extend Colab's idle or 12 h limits.
        while not _supervisor_stop.wait(BEAT_SECONDS):
            supervisor_tick()

    _supervisor_thread = threading.Thread(target=loop, name='rgm-supervisor', daemon=True)
    _supervisor_thread.start()
    wlog(f'supervisor started, beat {BEAT_SECONDS}s')

def supervisor_stop():
    _supervisor_stop.set()
    wlog('supervisor stopped')

def supervisor_status():
    age = int((time.time() - worker['started_at']) / 60)
    probe = worker.get('last_probe') or {}
    print(f"worker: kernel {worker['kernel']}, up {age} min, {worker['beats']} beats, "
          f"{worker['restarts']} restarts")
    print(f"worker:   comfy {comfy_ready()}, armed {worker.get('comfy_armed')}, "
          f"vram {globals().get('RUNNING_VRAM') or globals().get('VRAM_MODE')}")
    print(f"worker:   caddy {_alive('tunnel_caddy_proc')}, tailscaled {_alive('ts_proc')}, "
          f"cloudflared {_alive('tunnel_cf_proc')}")
    if probe:
        print(f"worker:   last public probe {_stamp(probe.get('at'))} "
              f"{'ok ' + str(probe.get('ms')) + ' ms' if probe.get('ok') else 'FAILED'}")
    if worker.get('last_error'):
        print('worker:   last error', worker['last_error'])

worker_report()
if LIFECYCLE:
    supervisor_start()
    print(f'worker: status file {STATUS_PATH}')
    print('worker: keep-alive = health probe + restart of dead children + state mirror, '
          'every', f'{BEAT_SECONDS}s')
else:
    print('worker: supervisor off. Nothing restarts ComfyUI if it dies.')


## 4 · Start ComfyUI (one process)
Low VRAM is the tested default. Switching memory mode restarts ComfyUI only. On MOSS, system RAM may be the bottleneck; normal VRAM is optional and can cause GPU OOM. Pause remote clients and wait for the queue to clear before changing modes.

In [ ]:

# @title Launch settings
VRAM_MODE = 'normalvram' # @param ['lowvram','normalvram']

if ACTIVE_PROFILE is None:
    raise RuntimeError('Prepare a model in section 3.')

# Restart ComfyUI if the model or VRAM mode has changed
if comfy_ready() and (
    globals().get('RUNNING_CHOICE') != MODEL_CHOICE
    or globals().get('RUNNING_VRAM') != VRAM_MODE
):
    stop_owned_server()

if 'start_comfy' not in globals():
    raise RuntimeError('Run section 3b (worker lifecycle) first; it defines start_comfy().')

if not start_comfy(VRAM_MODE):
    raise RuntimeError('ComfyUI failed to start; see the log above.')

RUNNING_CHOICE = MODEL_CHOICE
RUNNING_VRAM = VRAM_MODE

print('ComfyUI ready:', BASE_URL, '|', RUNNING_CHOICE)
print('VRAM mode:', RUNNING_VRAM)


## 5 · Shared API helpers
The API graph is built for the selected family, including the correct latent format, encoder, and VAE.

In [ ]:
import io, json, math, random, shutil, time
from pathlib import Path

import requests
from PIL import Image
from IPython.display import display, Markdown

BASE_URL = 'http://127.0.0.1:8188'
def ui_available():
    """True only when a browser is attached and allowed to answer a prompt."""
    if globals().get('AUTOMATION'):
        return False
    try:
        from google.colab import output
        return bool(output.eval_js('google.colab.kernel.accessAllowed'))
    except Exception:
        return True

def comfy_get(path, **kwargs):
    r = requests.get(BASE_URL + path, **kwargs)
    r.raise_for_status()
    return r.json()

def comfy_post(path, **kwargs):
    r = requests.post(BASE_URL + path, **kwargs)
    r.raise_for_status()
    return r.json()

def assert_comfy_running():
    r = requests.get(BASE_URL + '/system_stats', timeout=10)
    r.raise_for_status()
    return r.json()

def object_info():
    return comfy_get('/object_info', timeout=30)

def verify_required_nodes():
    info=object_info(); p=ACTIVE_PROFILE
    required=['UNETLoader','CLIPLoader','VAELoader','CLIPTextEncode',p['latent'],'KSampler','VAEDecode','SaveImage','LoraLoaderModelOnly']
    missing=[n for n in required if n not in info]
    if missing: raise RuntimeError('Missing nodes: '+str(missing))
    if p['clip_type'] not in info['CLIPLoader']['input']['required']['type'][0]: raise RuntimeError('Update ComfyUI: selected encoder type missing.')
    print('Native nodes verified for',MODEL_CHOICE)

def build_workflow(profile, positive, negative, width=768,height=768,steps=None,cfg=None,seed=-1,sampler=None,scheduler=None,loras=(),loader='LoraLoaderModelOnly',clip_device='default',use_reference=False,reference_filename=None,allow_experimental=False,ref_guidance=4.5,adapter_strength=1.0,ref_end=0.6,ref_megapixels=0.5,prefix='unified/image'):
    if min(width,height)<256 or width%16 or height%16: raise ValueError('Dimensions must be >=256 and multiples of 16.')
    if not positive.strip():raise ValueError('Enter a prompt.')
    if steps is not None and steps<1:raise ValueError('Steps must be positive.')
    if cfg is not None and cfg<0:raise ValueError('CFG must be nonnegative.')
    if use_reference:
        if profile['family']!='anima':raise ValueError('Anima Reference Adapter is not compatible with MOSS.')
        if profile['checkpoint']!='anima-base-v1.0.safetensors' and not allow_experimental:raise ValueError('Reference Adapter is trained on Anima Base v1.0. Experimental override required for other variants.')
        if not reference_filename:raise ValueError('Upload reference image first.')
    seed=random.randrange(2**63) if seed<0 else int(seed)
    g={}
    def node(i,kind,**inputs):
        g[str(i)]={'class_type':kind,'inputs':inputs};return [str(i),0]
    m=node(1,'UNETLoader',unet_name=profile['checkpoint'],weight_dtype='default')
    clip=node(2,'CLIPLoader',clip_name=profile['encoder'],type=profile['clip_type'],device=clip_device)
    v=node(3,'VAELoader',vae_name=profile['vae'])
    for i,(name,weight) in enumerate(loras,100):
        if float(weight)!=0:m=node(i,loader,model=m,lora_name=name,strength_model=float(weight))
    if use_reference:
        image=node(10,'LoadImage',image=reference_filename)
        m=node(11,'AnimaRefAdapter',model=m,vae=v,image=image,lora_name='anima_ref_adapter_v2_r64.safetensors',ref_guidance=float(ref_guidance),correction='apg',strength=float(adapter_strength),ref_end=float(ref_end),ref_megapixels=float(ref_megapixels))
    pos=node(4,'CLIPTextEncode',clip=clip,text=positive)
    neg=node(5,'CLIPTextEncode',clip=clip,text=negative)
    latent=node(6,profile['latent'],width=int(width),height=int(height),batch_size=1)
    sampled=node(7,'KSampler',model=m,positive=pos,negative=neg,latent_image=latent,seed=seed,steps=int(profile['steps'] if steps is None else steps),cfg=float(profile['cfg'] if cfg is None else cfg),sampler_name=sampler or profile['sampler'],scheduler=scheduler or profile['scheduler'],denoise=1.0)
    decoded=node(8,'VAEDecode',samples=sampled,vae=v)
    node(9,'SaveImage',images=decoded,filename_prefix=prefix)
    return g,seed

def queue_prompt(prompt):
    try:
        result = comfy_post('/prompt', json={'prompt': prompt}, timeout=120)
    except requests.HTTPError as exc:
        raise RuntimeError('ComfyUI rejected workflow: '+exc.response.text[:2200]) from exc
    return result['prompt_id']

def wait_for_completion(prompt_id, timeout=1800, poll=2):
    started = time.time()
    while time.time() - started < timeout:
        history = comfy_get(f'/history/{prompt_id}', timeout=60)
        if prompt_id in history:
            return history[prompt_id]
        time.sleep(poll)
    raise TimeoutError(f'Prompt {prompt_id} did not finish within {timeout} seconds.')

def extract_images_from_history(history_entry):
    images = []
    outputs = history_entry.get('outputs', {})
    for node_id, node_output in outputs.items():
        for img in node_output.get('images', []):
            images.append(img)
    return images

def fetch_image(image_meta):
    params = {
        'filename': image_meta['filename'],
        'subfolder': image_meta.get('subfolder', ''),
        'type': image_meta.get('type', 'output'),
    }
    r = requests.get(BASE_URL + '/view', params=params, timeout=120)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert('RGB')

def maybe_copy_output_to_cache(image_meta):
    if not USE_DRIVE:
        return None
    src = COMFY / 'output' / image_meta.get('subfolder', '') / image_meta['filename']
    dst_dir = CACHE / 'outputs' / ACTIVE_PROFILE['family']
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / image_meta['filename']
    if src.exists():
        shutil.copy2(src, dst)
        return dst
    return None

assert_comfy_running()
verify_required_nodes()

## 6 · Optional Anima reference image
Disabled by default. MOSS cannot use the ArcAnima adapter. For Anima Base v1.0, upload one reference here then enable the adapter in section 9. Other Anima versions require an experimental override.

In [ ]:
# @title Upload reference (optional)
UPLOAD_REFERENCE=False # @param {type:"boolean"}
REFERENCE_LOCAL_PATH=globals().get('REFERENCE_LOCAL_PATH')
if UPLOAD_REFERENCE:
    if ACTIVE_PROFILE['family']!='anima': raise ValueError('Select Anima for the Reference Adapter.')
    if not ui_available():
        raise RuntimeError('The file picker needs a browser. Set REFERENCE_LOCAL_PATH to a file '
                           'that already exists (Drive, or uploaded by the automation) instead.')
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Upload exactly one reference image.')
    name,payload=next(iter(uploaded.items()))
    ref=Path('/content/unified_reference.png')
    with Image.open(io.BytesIO(payload)) as im:
        im.convert('RGB').save(ref)
    REFERENCE_LOCAL_PATH=str(ref)
    del uploaded,payload
    gc.collect()
    display(Image.open(ref))
else: print('Reference upload skipped.')

## 7 · Optional manual LoRA upload
For Civitai downloads that require browser authentication, download a compatible `.safetensors` file and upload it here. Use the exact filename in the next panel. Files are separated by model family; local uploads cannot be checked for architecture automatically. Use small files on Free Colab because uploads temporarily occupy notebook RAM.
`LORA_URLS` takes direct `.safetensors` links and works without a browser, which is what an automated run needs; the picker is off by default and refuses to open when `AUTOMATION` is on.


In [ ]:
# @title Upload or pin local LoRAs (optional)
UPLOAD_LOCAL_LORA=True # @param {type:"boolean"}
LORA_FILE="" # @param {type:"string"}
LORA_FILE_WEIGHT=0.8 # @param {type:"number"}
LORA_URLS="https://huggingface.co/axiomofmind/Moss-image-nsfw-uncensored-lora/resolve/main/MOSS-Image-NSFW-Uncensored-LoRA-ComfyUI.safetensors?download=true" # @param {type:"string"}
# The bundled default is MOSS-specific; do not download it for Anima.
if ACTIVE_PROFILE['family'] != 'moss' and LORA_URLS == 'https://huggingface.co/axiomofmind/Moss-image-nsfw-uncensored-lora/resolve/main/MOSS-Image-NSFW-Uncensored-LoRA-ComfyUI.safetensors?download=true':
    LORA_URLS = ''; UPLOAD_LOCAL_LORA = False
# Three ways in, in priority order:
#   LORA_FILE         one exact file: an absolute path, a filename on Drive, or a
#                     https link. Installed and pinned as the active LoRA - no picker.
#   LORA_URLS         https links; installed but not selected.
#   UPLOAD_LOCAL_LORA the browser picker. Off by default; refuses to run headless.
# LoRAs already on Drive are restored by section 3 and need none of this.
#
# A link must be the file itself, not a page that links to it:
#   Hugging Face   .../resolve/main/name.safetensors    (a /blob/ link is rewritten)
#   Civitai        https://civitai.com/api/download/models/<versionId>
#                  needs CIVITAI_TOKEN in MyDrive/rgm-tailnet/credentials.json
# Anything that is not a SafeTensors file is rejected before it reaches the parser,
# quoting the first bytes of the answer so the cause is visible.

import re, struct
from urllib.parse import urlparse, parse_qs

def _civitai_token():
    for name in ('CIVITAI_TOKEN', 'CIVITAI_API_TOKEN'):
        value = os.environ.get(name, '').strip()
        if value:
            return value
    try:
        from google.colab import userdata
        return (userdata.get('CIVITAI_TOKEN') or '').strip()
    except Exception:
        return ''

def _redact(url, token):
    return url.replace(token, '<token>') if token else url

def _prepare_url(url):
    """Turn a known page link into a file link; attach the Civitai token."""
    host = (urlparse(url).hostname or '').lower()
    if 'civitai.com' in host and '/api/download/' not in url:
        version_id = (parse_qs(urlparse(url).query).get('modelVersionId') or [None])[0]
        if not version_id:
            raise ValueError(
                'That is a model page, not a file. Copy the version download link '
                '(https://civitai.com/api/download/models/<id>) or use section 8, '
                'which resolves model pages for you.')
        url = f'https://civitai.com/api/download/models/{version_id}'
        print('Note: model page resolved to', url)
    if 'huggingface.co' in host and '/blob/' in url:
        url = url.replace('/blob/', '/resolve/')
        print('Note: /blob/ is a web page; using the /resolve/ file link instead.')
    token = _civitai_token() if 'civitai.com' in host else ''
    if token and 'token=' not in url:
        url += ('&' if '?' in url else '?') + 'token=' + token
    return url, token

def _url_name(url):
    return Path(urlparse(url).path).name or 'lora.safetensors'

def _served_name(response):
    disposition = response.headers.get('Content-Disposition', '')
    match = re.search(r"filename\*?=(?:UTF-8''|[\"']?)([^\"';]+)", disposition, re.I)
    return match.group(1).strip() if match else ''

def _peek(path):
    return path.read_bytes()[:180].decode('utf-8', 'replace').strip().replace('\n', ' ')[:160]

def _check_safetensors(path, label=None):
    """Reject anything that is not a SafeTensors file, naming the reason."""
    name = label or path.name
    size = path.stat().st_size
    problem = None
    if size < 16:
        problem = f'only {size} bytes'
    else:
        with open(path, 'rb') as handle:
            header_length = struct.unpack('<Q', handle.read(8))[0]
            if not 0 < header_length < 100_000_000 or header_length + 8 > size:
                problem = f'header length {header_length} does not fit a {size}-byte file'
            else:
                handle.seek(8)
                if handle.read(1) != b'{':
                    problem = 'header is not JSON'
    if problem:
        raise RuntimeError(
            f'{name} is not a SafeTensors file ({problem}). '
            f'First bytes of the answer: {_peek(path)!r}. '
            'The link returned a web page, a login redirect or an error body.')
    try:
        with safe_open(str(path), framework='np') as handle:
            keys = list(handle.keys())
    except Exception as exc:
        raise RuntimeError(f'{name} failed SafeTensors parsing: {exc}') from exc
    if not keys:
        raise ValueError(f'{name} is an empty SafeTensors file')

def _fetch(response, dest):
    expected = int(response.headers.get('Content-Length') or 0)
    written = 0
    with open(dest, 'wb') as out:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                out.write(chunk)
                written += len(chunk)
    if expected and written != expected:
        raise IOError(f'truncated download: {written} of {expected} bytes')
    return written

def _mirror(dest):
    if USE_DRIVE:
        shutil.copy2(dest, LORA_CACHE / dest.name)
    print('Installed:', dest.name, f'({dest.stat().st_size / 1048576:.1f} MiB)')
    return dest

def _install_lora(name, origin):
    """Install one LoRA and return the installed path.

    `origin` is bytes (picker), a Path (already on disk) or an https URL. A URL does
    not always carry a filename (Civitai /api/download/models/NNN), so the server's
    Content-Disposition decides the name once it answers.
    """
    if not isinstance(origin, str):
        dest = LORA_DIR / Path(name).name
        temp = dest.with_suffix('.part')
        try:
            if isinstance(origin, bytes):
                temp.write_bytes(origin)
            else:
                shutil.copy2(origin, temp)
            _check_safetensors(temp, dest.name)
            temp.replace(dest)
        finally:
            temp.unlink(missing_ok=True)
        return _mirror(dest)

    url, token = _prepare_url(origin)
    shown = _redact(url, token)
    host = (urlparse(url).hostname or '').lower()
    temp = LORA_DIR / '.download.part'
    try:
        with requests.get(url, stream=True, timeout=(30, 300)) as response:
            if response.status_code in (401, 403):
                raise PermissionError(
                    f'{host} refused the download ({response.status_code}). Civitai needs '
                    'a token: add CIVITAI_TOKEN to MyDrive/rgm-tailnet/credentials.json or '
                    'to Colab secrets, or download it in a browser and use the picker.')
            response.raise_for_status()
            content_type = response.headers.get('Content-Type', '').lower()
            if content_type.startswith('text/') or 'json' in content_type:
                body = next(response.iter_content(160), b'')
                raise RuntimeError(
                    f'{host} answered with {content_type}, not a model file. '
                    f'First bytes: {body[:160]!r}. {shown} is a page or a redirect, '
                    'not a direct download link.')
            name = _served_name(response) or name
            if not name.lower().endswith('.safetensors'):
                if '/api/download/models/' in url:
                    name = 'civitai_' + url.split('/api/download/models/')[1].split('?')[0] + '.safetensors'
                else:
                    raise ValueError(f'{shown} served {name!r}, not a .safetensors file.')
            dest = LORA_DIR / Path(name).name
            temp = dest.with_suffix('.part')
            _fetch(response, temp)
        _check_safetensors(temp, dest.name)
        temp.replace(dest)
        return _mirror(dest)
    finally:
        temp.unlink(missing_ok=True)

def _find_lora(value):
    """Locate an existing LoRA by absolute path, installed name, or Drive cache name."""
    path = Path(value)
    if path.is_file():
        return path
    for candidate in (LORA_DIR / path.name, LORA_CACHE / path.name):
        if candidate.is_file():
            return candidate
    return None

def _pin_lora(value, weight):
    """Install if needed, then return the entry the generation cell will validate."""
    name = _url_name(value) if value.startswith('http') else Path(value).name
    if not value.startswith('http') and not name.lower().endswith('.safetensors'):
        raise ValueError(f'{value} is not a .safetensors file.')
    # A local value may be an absolute path; a URL only gives us its last segment.
    existing = _find_lora(name if value.startswith('http') else value)
    if existing is not None:
        dest = LORA_DIR / existing.name
        same = dest.is_file() and existing.stat().st_size == dest.stat().st_size
        if same:
            print('Already installed:', existing.name)
        else:
            dest = _install_lora(existing.name, existing)   # copy, re-validate, replace
        if USE_DRIVE and not (LORA_CACHE / dest.name).is_file():
            shutil.copy2(dest, LORA_CACHE / dest.name)
    elif value.startswith('http'):
        dest = _install_lora(name, value)
    else:
        raise FileNotFoundError(
            f'{value} not found. Looked at that path, {LORA_DIR / name}, '
            f'and {LORA_CACHE / name}.')
    print(f'Pinned: {dest.name} at weight {weight}')
    return dest.name, float(weight)

PINNED_LORA = None
if LORA_FILE.strip():
    PINNED_LORA = _pin_lora(LORA_FILE.strip(), float(LORA_FILE_WEIGHT))
elif LORA_URLS.strip():
    for link in [u.strip() for u in LORA_URLS.replace('\n', ',').split(',') if u.strip()]:
        try:
            _install_lora(_url_name(link), link)
        except Exception as exc:
            print('Failed:', _url_name(link), '->', str(exc)[:300])
elif UPLOAD_LOCAL_LORA:
    if not ui_available():
        raise RuntimeError('The file picker needs a browser. Set LORA_FILE or put the files in '
                           'MyDrive/Unified_Image_Assets/loras/<family>/ instead.')
    from google.colab import files
    uploaded=files.upload()
    for original,data in uploaded.items():
        name=Path(original).name
        if not name.lower().endswith('.safetensors'):print('Skipped:',name);continue
        _install_lora(name, data)
    del uploaded
    if 'data' in globals():del data
    gc.collect()
    LORA_CONFIG=[];LORA_VALIDATED_SIGNATURE=None
else:print('Upload skipped. LoRAs restored from Drive stay installed.')

## 8 · Weighted Civitai LoRA manager
Paste ordinary Civitai model URLs (`.com` or `.red`), explicit version links, or direct Civitai download links. The manager finds a matching Anima or MOSS version if no version is specified, validates explicit versions, checks SafeTensors and hashes when available, and streams the file rather than storing its bytes in Python. Individual weights are preserved. Local filenames are supported. Civitai may still deny access; use the optional token or manual upload. Rerun this panel after model switches. Keep compatible LoRAs small and use only a few on MOSS. Anima variants share a family, but individual LoRAs may behave differently on Turbo/Aesthetic.

In [ ]:
# @title Weighted LoRA manager
AUTO_LORAS="" # @param {type:"string"}
# "name.safetensors:0.8, https://civitai.com/...:0.6" pins the stack with no
# clicking. Leave empty to use LORA_FILE from section 7, or to use the panel.
import re, struct, hashlib, ipywidgets as widgets
from urllib.parse import urlparse, parse_qs
from IPython.display import display, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass

lora_rows=[]
LORA_CONFIG=[]
LORA_VALIDATED_FAMILY=None
LORA_VALIDATED_SIGNATURE=None
PANEL_FAMILY=ACTIVE_PROFILE['family']
lora_message=widgets.Output()
rows_box=widgets.VBox([])
status=widgets.HTML('<b>LoRAs:</b> none selected')
token_input=widgets.Password(description='Civitai token:',placeholder='Optional; never shared',style={'description_width':'100px'},layout=widgets.Layout(width='440px'))
add_btn=widgets.Button(description='Add LoRA',icon='plus')
apply_btn=widgets.Button(description='Download / validate',button_style='success',icon='download')

# Use the API for precise version detection and provenance, not page scraping.
CIVITAI_HOSTS={'civitai.com','www.civitai.com','civitai.red','www.civitai.red'}

def civitai_get_json(endpoint,headers):
    r=requests.get('https://civitai.com'+endpoint,headers=headers,timeout=60)
    if r.status_code in (401,403): raise PermissionError('Civitai metadata requires authorization or is restricted.')
    r.raise_for_status()
    return r.json()

def compatible_base(label):
    normalized=re.sub(r'[^a-z0-9]','',str(label).lower())
    return (normalized.startswith('mossimage') or normalized=='moss') if ACTIVE_PROFILE['family']=='moss' else normalized.startswith('anima')

def resolve_civitai(source,headers):
    parsed=urlparse(source)
    if parsed.scheme!='https' or parsed.hostname not in CIVITAI_HOSTS:
        raise ValueError('Use a Civitai HTTPS model page, version link, direct Civitai download URL, or installed filename.')
    q=parse_qs(parsed.query)
    version_id=(q.get('modelVersionId') or [None])[0]
    m=re.search(r'/api/download/models/(\d+)',parsed.path) or re.search(r'/model-versions/(\d+)',parsed.path)
    if m: version_id=m.group(1)
    if not version_id:
        match=re.search(r'/models/(\d+)',parsed.path)
        if not match: raise ValueError('Model URL missing /models/ID.')
        model=civitai_get_json(f'/api/v1/models/{match.group(1)}',headers)
        candidates=[v for v in model.get('modelVersions',[]) if compatible_base(v.get('baseModel'))]
        if not candidates:
            available=', '.join(f"{v.get('name')} ({v.get('baseModel')})" for v in model.get('modelVersions',[]))
            raise ValueError('No compatible version for the selected model family. Available: '+available[:700])
        # Civitai typically returns newest first. Prefer exact base model MOSS-Image.
        chosen=next((v for v in candidates if compatible_base(v.get('baseModel'))),candidates[0])
        version_id=str(chosen['id'])
        print('Automatically selected compatible version:',chosen.get('name'),version_id)
    version=civitai_get_json(f'/api/v1/model-versions/{int(version_id)}',headers)
    model=version.get('model',{})
    if model.get('type','').lower()!='lora':
        raise ValueError('Civitai resource type is not LoRA: '+str(model.get('type')))
    if not compatible_base(version.get('baseModel')):
        raise ValueError(f"Version {version_id} targets {version.get('baseModel')}, not the selected model family. Choose a compatible version.")
    files=[f for f in version.get('files',[]) if f.get('name','').lower().endswith('.safetensors')]
    if not files: raise ValueError('This version has no public .safetensors file.')
    wanted=(q.get('fileId') or [None])[0]
    selected=next((f for f in files if str(f.get('id'))==str(wanted)),None) if wanted else None
    if wanted and selected is None: raise ValueError('Requested fileId is not a SafeTensors file in this version.')
    selected=selected or next((f for f in files if f.get('primary')),files[0])
    url=selected.get('downloadUrl') or version.get('downloadUrl')
    if not url: raise ValueError('Civitai has no accessible download URL for this version.')
    print('LoRA:',model.get('name'), '| version:',version.get('name'),'| base:',version.get('baseModel'))
    return version,selected,url

def check_safetensors(path):
    with safe_open(str(path),framework='np') as handle:
        if not list(handle.keys()):raise ValueError('Empty SafeTensors file')

def download_lora(source,headers):
    if not source.startswith('https://'):
        name=Path(source).name
        if name.startswith('anima_ref_adapter'):raise ValueError('Use the dedicated Reference Adapter switch instead.')
        if source not in (name, ACTIVE_PROFILE['family']+'/'+name): raise ValueError('Select a local LoRA from the active family.')
        if not (LORA_DIR/name).is_file():
            cached=LORA_CACHE/name
            if not cached.is_file():
                raise FileNotFoundError('LoRA not installed: '+source)
            shutil.copy2(cached,LORA_DIR/name)
            print('Restored from cache:',name)
        check_safetensors(LORA_DIR/name)
        return ACTIVE_PROFILE['family']+'/'+name
    version, selected, url = resolve_civitai(source,headers)
    # Prefix version+file IDs to avoid collisions across models with identical filenames.
    name=f"civitai_{version['id']}_{selected['id']}_{Path(selected['name']).name}"
    destination=LORA_DIR/name
    cached=LORA_CACHE/name
    cached.parent.mkdir(parents=True,exist_ok=True)
    if destination.is_file() and destination.stat().st_size>1024:
        check_safetensors(destination)
        print('Already installed:',name)
        return ACTIVE_PROFILE['family']+'/'+name
    if cached.is_file() and cached.stat().st_size>1024:
        shutil.copy2(cached,destination)
        check_safetensors(destination)
        print('Restored from cache:',name)
        return ACTIVE_PROFILE['family']+'/'+name
    temp=destination.with_name(destination.name+'.part')
    try:
        with requests.get(url,headers=headers if urlparse(url).hostname in CIVITAI_HOSTS else {},stream=True,timeout=180) as r:
            if r.status_code in (401,403):
                raise PermissionError('Civitai refused this file (401/403). Enter your Civitai token or download the compatible file in your browser and upload it manually.')
            r.raise_for_status()
            if 'text/html' in r.headers.get('Content-Type','').lower() or 'application/json' in r.headers.get('Content-Type','').lower():
                raise RuntimeError('Civitai returned an error page, not a model file.')
            with open(temp,'wb') as f:
                for chunk in r.iter_content(1024*1024):
                    if chunk: f.write(chunk)
        check_safetensors(temp)
        expected=(selected.get('hashes') or {}).get('SHA256')
        if expected and len(expected)==64:
            sha=hashlib.sha256()
            with open(temp,'rb') as f:
                for chunk in iter(lambda:f.read(4*1024*1024),b''): sha.update(chunk)
            if sha.hexdigest().lower()!=expected.lower(): raise ValueError('Downloaded file failed SHA256 verification.')
        temp.replace(destination)
        if USE_DRIVE: shutil.copy2(destination,cached)
        print('Installed:',name, f'({destination.stat().st_size/1048576:.1f} MiB)')
    finally:
        temp.unlink(missing_ok=True)
    return ACTIVE_PROFILE['family']+'/'+name

def invalidate(_=None):
    global LORA_CONFIG,LORA_VALIDATED_FAMILY,LORA_VALIDATED_SIGNATURE
    LORA_CONFIG=[]
    LORA_VALIDATED_FAMILY=None
    LORA_VALIDATED_SIGNATURE=None
    status.value='<b>Changed:</b> click Download / validate before generating.'

def render_rows():
    rows_box.children=tuple(widgets.HBox([r['source'],r['strength'],r['enabled'],r['remove']]) for r in lora_rows)

def add_row(source='',weight=0.8,enabled=True):
    field=widgets.Text(value=source,description='LoRA:',placeholder='Civitai page URL / version URL / filename',layout=widgets.Layout(width='65%'),style={'description_width':'50px'})
    strength=widgets.FloatText(value=weight,description='Weight:',layout=widgets.Layout(width='145px'),style={'description_width':'65px'})
    checked=widgets.Checkbox(value=enabled,description='Use',indent=False,layout=widgets.Layout(width='70px'))
    remove=widgets.Button(description='Remove',icon='trash')
    row={'source':field,'strength':strength,'enabled':checked,'remove':remove}
    lora_rows.append(row)
    def remove_it(_):
        lora_rows.remove(row);render_rows();invalidate()
    remove.on_click(remove_it)
    for w in (field,strength,checked):w.observe(invalidate,names='value')
    render_rows();invalidate()

def active_signature():
    return tuple((r['source'].value.strip(),float(r['strength'].value)) for r in lora_rows if r['enabled'].value and r['source'].value.strip())

def prepare_loras(_=None):
    global LORA_CONFIG,LORA_VALIDATED_FAMILY,LORA_VALIDATED_SIGNATURE
    LORA_CONFIG=[]
    LORA_VALIDATED_FAMILY=None
    LORA_VALIDATED_SIGNATURE=None
    with lora_message:
        clear_output(wait=True)
        active=[r for r in lora_rows if r['enabled'].value and r['source'].value.strip()]
        headers={'Authorization':'Bearer '+token_input.value.strip()} if token_input.value.strip() else {}
        prepared=[]
        try:
            if PANEL_FAMILY!=ACTIVE_PROFILE['family']:raise RuntimeError('Rerun the LoRA panel after switching model families.')
            if comfy_ready():require_idle()
            for r in active:
                weight=float(r['strength'].value)
                if not -5<=weight<=5: raise ValueError('LoRA strength outside -5..5')
                prepared.append((download_lora(r['source'].value.strip(),headers),weight))
            LORA_CONFIG=prepared
            LORA_VALIDATED_FAMILY=ACTIVE_PROFILE['family']
            LORA_VALIDATED_SIGNATURE=active_signature()
            status.value=f'<b>Ready:</b> {len(prepared)} LoRA(s)'
            print('Ready:',LORA_CONFIG)
        except Exception as e:
            status.value='<b>Failed:</b> inspect the error below'
            print(type(e).__name__+':',str(e).split(' for url:')[0])
        finally:
            token_input.value=''
            # Keep empty config; generation fails safely rather than using stale weights.

def parse_auto(raw):
    """'name:0.8, url:0.6' -> [(name, 0.8), (url, 0.6)]; no float means 0.8."""
    entries = []
    for item in raw.replace('\n', ',').split(','):
        item = item.strip()
        if not item:
            continue
        source, sep, tail = item.rpartition(':')
        try:
            entries.append((source, float(tail)) if sep else (item, 0.8))
        except ValueError:
            entries.append((item, 0.8))
    return entries

_auto = (parse_auto(AUTO_LORAS) if AUTO_LORAS.strip()
         else ([globals()['PINNED_LORA']] if globals().get('PINNED_LORA') else []))
if _auto:
    lora_rows.clear()
    for _source, _weight in _auto:
        add_row(_source, _weight, True)
    render_rows()
    prepare_loras()

add_btn.on_click(lambda _:add_row())
apply_btn.on_click(prepare_loras)
add_row()
add_row(enabled=False)
display(widgets.VBox([widgets.HTML('<h3>Weighted '+ACTIVE_PROFILE['family'].upper()+' LoRAs — regular Civitai URLs</h3>'),rows_box,add_btn,token_input,apply_btn,status,lora_message]))

## 9 · Generation settings
Automatic presets follow the selected profile: MOSS 25 steps/CFG 2.5, Anima Base/Aesthetic 30/4.5, Turbo 10/1.0. Use override values only if desired. Resolution 768×768, batch size 1. For memory issues try 512×768; first generation after a switch is slower. Reference defaults use 0.5 MP and ref_end 0.6 to reduce cost; increase if needed.

In [ ]:
# @title Generation controls
POSITIVE_PROMPT='A cinematic 1990s anime illustration of a quiet cafe at dusk, warm light through the window, hand-painted backgrounds, rich shadows.' # @param {type:"string"}
NEGATIVE_PROMPT='diagonal streaks, speed lines, repeated stripes, banding, glitch texture, motion blur, distorted pattern,  worst quality, low quality, text, logo, bad hands, missing fingers, multiple limbs, bad anatomy, bobbles,' # @param {type:"string"}
WIDTH=1024 # @param {type:"integer"}
HEIGHT=1024 # @param {type:"integer"}
USE_MODEL_PRESET=True # @param {type:"boolean"}
STEPS=25 # @param {type:"integer"}
CFG=2.5 # @param {type:"number"}
SEED=-1 # @param {type:"integer"}
SAMPLER='er_sde' # @param ['er_sde','euler','dpmpp_2m']
SCHEDULER='simple' # @param ['simple','normal','beta']
CLIP_DEVICE='default' # @param ['default','cpu']
LORA_LOADER='LoraLoaderModelOnly' # @param ['LoraLoaderModelOnly','LoraLoaderBypassModelOnly']
USE_REFERENCE_ADAPTER=False # @param {type:"boolean"}
ALLOW_EXPERIMENTAL_ADAPTER=False # @param {type:"boolean"}
REF_GUIDANCE=4.5 # @param {type:"number"}
ADAPTER_STRENGTH=1.0 # @param {type:"number"}
REF_END=0.6 # @param {type:"number"}
REF_MEGAPIXELS=0.5 # @param {type:"number"}
if USE_REFERENCE_ADAPTER and ACTIVE_PROFILE['family']!='anima':raise ValueError('Disable Reference Adapter for MOSS.')
if USE_REFERENCE_ADAPTER and ACTIVE_PROFILE['checkpoint']!='anima-base-v1.0.safetensors' and not ALLOW_EXPERIMENTAL_ADAPTER:raise ValueError('Select Anima Base v1.0 or explicitly enable experimental adapter use.')
print('Active model:',MODEL_CHOICE,'| resolution:',WIDTH,'x',HEIGHT)
print('Steps / CFG:',(ACTIVE_PROFILE['steps'],ACTIVE_PROFILE['cfg']) if USE_MODEL_PRESET else (STEPS,CFG))

## 10 · Generate inline and export API workflow
The last successful generation graph is saved for Lumiverse. Keep installed filenames unchanged on the backend; family-specific LoRAs appear as `anima/...` or `moss/...` in the API graph. A remote client should import the correct model-specific workflow. Set `GENERATE = False` to skip submission entirely; nothing is queued and the previously saved API workflow is left untouched.

In [ ]:
# @title Generate inline and export API workflow
GENERATE=False # @param {type:"boolean"}
if GENERATE:
    assert_comfy_running()
    if RUNNING_CHOICE!=MODEL_CHOICE or ACTIVE_PROFILE is None:raise RuntimeError('Prepare and start the selected model first.')
    p=ACTIVE_PROFILE
    if 'lora_rows' not in globals() or PANEL_FAMILY!=p['family']:raise RuntimeError('Run the LoRA manager for this family first.')
    if active_signature() and (LORA_VALIDATED_FAMILY!=p['family'] or LORA_VALIDATED_SIGNATURE!=active_signature()):raise RuntimeError('Click Download / validate after selecting LoRAs.')
    loras=list(LORA_CONFIG) if active_signature() else []
    info=object_info()
    if loras and LORA_LOADER not in info:raise RuntimeError('Selected LoRA loader unavailable; use LoraLoaderModelOnly.')
    # Check installed assets and registered backend LoRA names before submitting.
    for name,weight in loras:
        if not (MODELS/'loras'/name).is_file():raise FileNotFoundError(name)
    ref_name=None
    if USE_REFERENCE_ADAPTER:
        if p['family']!='anima':raise ValueError('Reference Adapter cannot be used with MOSS.')
        if 'AnimaRefAdapter' not in info:raise RuntimeError('Install adapter in section 2 and restart ComfyUI.')
        if not globals().get('REFERENCE_LOCAL_PATH') or not Path(REFERENCE_LOCAL_PATH).is_file():raise RuntimeError('Upload reference in section 6.')
        with open(REFERENCE_LOCAL_PATH,'rb') as f:
            result=comfy_post('/upload/image',files={'image':('unified_reference.png',f,'image/png')},data={'type':'input','overwrite':'true'},timeout=120)
        ref_name=(result.get('subfolder','')+'/' if result.get('subfolder') else '')+result['name']
    workflow,actual_seed=build_workflow(p,POSITIVE_PROMPT,NEGATIVE_PROMPT,width=WIDTH,height=HEIGHT,steps=None if USE_MODEL_PRESET else STEPS,cfg=None if USE_MODEL_PRESET else CFG,seed=SEED,sampler=None if USE_MODEL_PRESET else SAMPLER,scheduler=None if USE_MODEL_PRESET else SCHEDULER,loras=loras,loader=LORA_LOADER,clip_device=CLIP_DEVICE,use_reference=USE_REFERENCE_ADAPTER,reference_filename=ref_name,allow_experimental=ALLOW_EXPERIMENTAL_ADAPTER,ref_guidance=REF_GUIDANCE,adapter_strength=ADAPTER_STRENGTH,ref_end=REF_END,ref_megapixels=REF_MEGAPIXELS,prefix=p['family']+'/image')
    print('Submitting:',MODEL_CHOICE,'seed:',actual_seed,'LoRAs:',loras)
    prompt_id=queue_prompt(workflow)
    print('Prompt ID:',prompt_id)
    record=wait_for_completion(prompt_id,timeout=3600)
    if record.get('status',{}).get('status_str')=='error':raise RuntimeError(str(record['status'])[-4000:])
    images=extract_images_from_history(record)
    if not images:raise RuntimeError('No images returned. Inspect server log below.')
    workflow_path=Path('/content/unified_workflow_api.json');workflow_path.write_text(json.dumps(workflow,indent=2))
    family_path=Path('/content/'+p['family']+'_workflow_api.json');family_path.write_text(json.dumps(workflow,indent=2))
    for item in images:
        display(fetch_image(item));print('Saved:',item['filename'])
        copied=maybe_copy_output_to_cache(item)
        if copied:print('Drive:',copied)
    print('API workflow:',family_path)
else:print('Generation skipped (GENERATE is off). Nothing was submitted.')


In [ ]:
# @title Optional: download last successful workflow
DOWNLOAD_WORKFLOW=False # @param {type:"boolean"}
if DOWNLOAD_WORKFLOW:
    from google.colab import files
    path=Path('/content/unified_workflow_api.json')
    if path.is_file():files.download(str(path))
    else:print('Generate an image first.')
else:print('Workflow download skipped.')

## 11 · Optional authenticated tunnel
For hosts that permit remote tunneling. On managed Colab, external proxies are restricted; keep this off on Free. This is connectivity only, not GPU provisioning. ComfyUI remains bound to localhost; a small Caddy reverse proxy adds HTTP Basic Auth in front of it, and a free Cloudflare quick tunnel (`trycloudflare.com`, no account, no payment) publishes it. The tunnel URL is ephemeral — copy it into the Lumiverse Space secrets after each change. Native Lumiverse needs the implementation-side gateway; the credentials are prompted privately and never written into workflow JSON. A successful health check from this notebook does not prove reachability from the Space.


In [ ]:
# @title Optional authenticated tunnel (Tailscale Funnel | cloudflared quick tunnel)
TUNNEL_MODE = "tailscale"   # @param ["tailscale", "cloudflare", "none"]
TS_HOSTNAME = "moss-comfy"  # @param {type:"string"}
TS_VERSION = "1.104.7"
PERSIST_TS_STATE = True     # @param {type:"boolean"}

import os, json, re, time, shutil, subprocess
from getpass import getpass
from pathlib import Path
import requests

UPSTREAM_PORT = 8189

if TUNNEL_MODE == "none":
    print("Tunnel disabled. Inline generation works without it.")
else:
    if not comfy_ready():
        raise RuntimeError("Start ComfyUI first.")

    # ---- stop whatever a previous run of this cell started ----
    for _name in ("tunnel_caddy_proc", "tunnel_cf_proc", "ts_proc"):
        _proc = globals().get(_name)
        if _proc is not None and _proc.poll() is None:
            _proc.terminate()
            try:
                _proc.wait(timeout=15)
            except Exception:
                _proc.kill()

    # ---- Caddy: Basic Auth in front of ComfyUI (identical for both tunnels) ----
    CADDY = Path("/content/caddy")
    if not CADDY.exists():
        subprocess.run(
            ["bash", "-c",
             "curl -L -o /content/caddy.tgz "
             "https://github.com/caddyserver/caddy/releases/download/v2.8.4/"
             "caddy_2.8.4_linux_amd64.tar.gz "
             "&& tar -xzf /content/caddy.tgz -C /content caddy "
             "&& rm /content/caddy.tgz"],
            check=True,
        )
        CADDY.chmod(0o755)

    def _secret(name, prompt):
        value = os.environ.get(name, "").strip()
        if not value:
            try:
                from google.colab import userdata
                value = (userdata.get(name) or "").strip()
            except Exception:
                value = ""
        if value:
            return value
        if globals().get('AUTOMATION'):
            raise RuntimeError(f'{name} is not set. Put it in Colab secrets (or the Space '
                               'environment) before running unattended.')
        return getpass(prompt).strip()

    # The bcrypt hash in the Caddyfile *is* the credential, and the Space already
    # carries the matching GATEWAY_BASIC_AUTH. Saving the file on Drive is what
    # stops the prompt every session. Delete it to rotate the password.
    SAVED_CADDYFILE = Path("/content/drive/MyDrive/rgm-tailnet/Caddyfile")
    BASIC_USER = BASIC_PASS = None
    if SAVED_CADDYFILE.is_file():
        shutil.copy2(SAVED_CADDYFILE, "/content/Caddyfile")
        print(f"Caddy: reusing the Basic Auth saved at {SAVED_CADDYFILE}")
    else:
        BASIC_USER = _secret("BASIC_AUTH_USER", "HTTP Basic Auth username: ")
        BASIC_PASS = _secret("BASIC_AUTH_PASS", "HTTP Basic Auth password: ")
        if not BASIC_USER or ":" in BASIC_USER or not BASIC_PASS:
            raise ValueError("Need a username (no colons) and a password.")
        _hashed = subprocess.check_output(
            [str(CADDY), "hash-password", "--plaintext", BASIC_PASS], text=True
        ).strip()
        Path("/content/Caddyfile").write_text(
            f":{UPSTREAM_PORT} {{\n"
            f"\tbasic_auth {{\n\t\t{BASIC_USER} {_hashed}\n\t}}\n"
            f"\treverse_proxy 127.0.0.1:8188\n"
            f"}}\n"
        )
        if SAVED_CADDYFILE.parent.is_dir():
            shutil.copy2("/content/Caddyfile", SAVED_CADDYFILE)
            print(f"Caddy: saved Basic Auth to {SAVED_CADDYFILE} for future sessions")
    tunnel_caddy_proc = subprocess.Popen(
        [str(CADDY), "run", "--config", "/content/Caddyfile"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(2)
    _probe = requests.get(
        f"http://127.0.0.1:{UPSTREAM_PORT}/system_stats",
        auth=(BASIC_USER, BASIC_PASS) if BASIC_PASS else None, timeout=10,
    )
    if BASIC_PASS:
        assert _probe.ok and "devices" in _probe.json(), "Local auth proxy health check failed."
    else:
        # Only the hash is on disk, so no credentials to send. 401 proves the proxy
        # is up and still enforcing auth; 200 would mean auth silently went away.
        assert _probe.status_code == 401, (
            f"Saved Caddyfile did not enforce Basic Auth (got {_probe.status_code}).")
    print(f"Caddy: http://127.0.0.1:{UPSTREAM_PORT} -> ComfyUI:8188 (Basic Auth on)")

    # ---------------- Tailscale Funnel ----------------
    if TUNNEL_MODE == "tailscale":
        TS = Path("/content/tailscale")
        TSD = Path("/content/tailscaled")
        if not (TS.is_file() and TSD.is_file()):
            _base = f"tailscale_{TS_VERSION}_amd64"
            subprocess.run(
                ["bash", "-c",
                 f"cd /content && curl -fsSL -o ts.tgz "
                 f"https://pkgs.tailscale.com/stable/{_base}.tgz "
                 f"&& tar -xzf ts.tgz "
                 f"&& install -m755 {_base}/tailscale /content/tailscale "
                 f"&& install -m755 {_base}/tailscaled /content/tailscaled "
                 f"&& rm -rf ts.tgz {_base}"],
                check=True,
            )

        # tailscaled keeps its node key (identity) and its TLS certificates in
        # --statedir, but it also creates a unix socket there. A unix socket
        # cannot be created on the Drive FUSE mount, and a state file on FUSE is
        # unreliable (flock, atomic rename). So it runs entirely on local disk,
        # and the identity files are mirrored to Drive once they are good.
        run_dir = Path("/content/ts-run")
        run_dir.mkdir(parents=True, exist_ok=True)
        sock = run_dir / "tailscaled.sock"

        persist_dir = None
        if PERSIST_TS_STATE and Path("/content/drive/MyDrive").is_dir():
            persist_dir = Path("/content/drive/MyDrive/rgm-tailnet")
            persist_dir.mkdir(parents=True, exist_ok=True)
            restored = []
            src_state = persist_dir / "tailscaled.state"
            if src_state.is_file() and src_state.stat().st_size > 0:
                shutil.copy2(src_state, run_dir / "tailscaled.state")
                restored.append("node identity")
            if (persist_dir / "certs").is_dir():
                shutil.copytree(persist_dir / "certs", run_dir / "certs", dirs_exist_ok=True)
                restored.append("TLS certificate")
            tail = f" - restored {', '.join(restored)}" if restored else " - first run, nothing to restore"
            print(f"Tailscale state: {persist_dir} (persisted){tail}")
        else:
            print("Tailscale state: EPHEMERAL (local disk only)")
            print("  WARNING: new node + new TLS certificate every session.")
            print("  Let's Encrypt allows ~5 duplicate certs/week per hostname;")
            print("  mount Drive (USE_DRIVE in section 2) to persist state instead.")

        def persist_ts_state(why):
            if persist_dir is None:
                return
            src = run_dir / "tailscaled.state"
            if src.is_file() and src.stat().st_size > 0:
                tmp = persist_dir / "tailscaled.state.tmp"
                shutil.copy2(src, tmp)
                os.replace(tmp, persist_dir / "tailscaled.state")
            if (run_dir / "certs").is_dir():
                shutil.copytree(run_dir / "certs", persist_dir / "certs", dirs_exist_ok=True)
            print(f"  persisted to Drive after {why}")

        # A restored state file is an already-registered node: the key was spent
        # on the first run and is not needed again. Only a fresh node asks for one.
        restored_login = (run_dir / "tailscaled.state").is_file()
        _key = os.environ.get("TS_AUTHKEY", "").strip()
        if not _key:
            try:
                from google.colab import userdata
                _key = (userdata.get("TS_AUTHKEY") or "").strip()
            except Exception:
                _key = ""
        if not _key and not restored_login:
            if globals().get('AUTOMATION'):
                raise RuntimeError('No saved tailnet identity and no TS_AUTHKEY. Set TS_AUTHKEY '
                                   'in Colab secrets for the first run against this Drive.')
            _key = getpass("Tailscale auth key (tskey-auth-...): ").strip()
        if not _key and not restored_login:
            raise RuntimeError("An auth key is required for a new node.")

        _log = open("/content/tailscaled.log", "w")
        # Kept as a variable so the supervisor can restart the daemon with the
        # exact same arguments instead of guessing them.
        TS_LAUNCH = [
            str(TSD),
            "--tun=userspace-networking",          # Colab has no /dev/net/tun
            f"--state={run_dir / 'tailscaled.state'}",
            f"--statedir={run_dir}",
            f"--socket={sock}",
            "--socks5-server=127.0.0.1:1055",
        ]
        ts_proc = subprocess.Popen(TS_LAUNCH, stdout=_log, stderr=subprocess.STDOUT)

        def ts_log_tail(n=3000):
            try:
                return Path("/content/tailscaled.log").read_text(errors="replace")[-n:]
            except Exception:
                return "(no log)"

        for _ in range(60):
            if sock.exists():
                break
            if ts_proc.poll() is not None:
                raise RuntimeError("tailscaled exited\n" + ts_log_tail())
            time.sleep(0.5)
        else:
            raise RuntimeError("tailscaled never created its control socket\n" + ts_log_tail())

        def tails(*args, check=True):
            r = subprocess.run([str(TS), "--socket", str(sock), *args],
                               text=True, capture_output=True)
            if check and r.returncode != 0:
                raise RuntimeError(("tailscale " + " ".join(args[:2]) + ": "
                                    + (r.stderr or r.stdout)).strip()[:1500])
            return r

        up_args = [f"--hostname={TS_HOSTNAME}", "--accept-dns=false", "--timeout=120s"]
        if _key:
            up_args.insert(0, f"--authkey={_key}")
        if not restored_login:
            # --reset drops prefs, including the funnel config; only a fresh node needs it.
            up_args.append("--reset")
        r = tails("up", *up_args, check=False)
        had_key = bool(_key)
        del _key
        if r.returncode != 0:
            hint = "" if had_key else " (the saved node may be logged out; set TS_AUTHKEY)"
            raise RuntimeError("tailscale up failed: "
                               + (r.stderr or r.stdout).strip()[:1400] + hint)

        status = {}
        for _ in range(90):
            _st = tails("status", "--json", check=False)
            try:
                status = json.loads(_st.stdout or "{}")
            except Exception:
                status = {}
            self_info = status.get("Self") or {}
            if status.get("BackendState") == "Running" and self_info.get("Online", True):
                break
            time.sleep(1)
        else:
            raise RuntimeError(f"Tailscale never reached Running "
                               f"(state={status.get('BackendState')})\n" + ts_log_tail())

        # A node can be renamed on registration (moss-comfy -> moss-comfy-1) when an
        # older node still holds the base name. `funnel --bg` ADDS to the serve config,
        # so an entry for the old hostname survives and the edge then refuses TLS for
        # both names. Clear the config first so this call is idempotent.
        tails("funnel", "reset", check=False)
        tails("serve", "reset", check=False)   # older clients keep funnel config here

        r = tails("funnel", "--bg", "--https=443", "--yes",
                  f"http://127.0.0.1:{UPSTREAM_PORT}", check=False)
        if r.returncode != 0:
            print((r.stderr or r.stdout).strip())
            raise RuntimeError(
                "funnel failed. Enable HTTPS certificates and Funnel in the "
                "Tailscale admin console first (see docs/spike-tailscale-funnel.md §2)."
            )

        dns = ((status.get("Self") or {}).get("DNSName") or "").rstrip(".")
        if not dns:
            _st = tails("status", "--json", check=False)
            dns = ((json.loads(_st.stdout or "{}").get("Self") or {}).get("DNSName") or "").rstrip(".")
        if not dns:
            raise RuntimeError("Could not read the node DNS name from tailscale status")

        # "Funnel on" in the local config is not proof the edge will serve it.
        # Surface any entry that points at a hostname this node does not own.
        _fs = tails("funnel", "status", check=False)
        _foreign = [ln.strip() for ln in (_fs.stdout or "").splitlines()
                    if "https://" in ln and dns not in ln]
        if _foreign:
            print("WARNING: funnel config still lists other hostnames:")
            for _ln in _foreign:
                print("   ", _ln)
            print("   They belong to other nodes; remove those nodes in the admin console.")

        # Force certificate issuance now, so a missing HTTPS setting fails here
        # with a readable error instead of silently after the Space connects.
        cert = tails("cert", "--cert-file", str(run_dir / "probe.crt"),
                     "--key-file", str(run_dir / "probe.key"), dns, check=False)
        if cert.returncode != 0:
            print("Certificate warning:", (cert.stderr or cert.stdout).strip()[:800])
        persist_ts_state("certificate issued")

        URL = f"https://{dns}"
        print()
        print("Funnel URL:", URL)
        print("  stable: identical every session while hostname + tailnet are unchanged")
        print()
        print("HF Space secrets - set ONCE, no restart in future sessions:")
        print("  GATEWAY_UPSTREAM=" + URL)
        if BASIC_PASS:
            print("  GATEWAY_BASIC_AUTH=" + BASIC_USER + ":<password you typed>")
        else:
            print("  GATEWAY_BASIC_AUTH=unchanged (reusing the saved Basic Auth)")
        print()
        print("First use of a new hostname: allow ~60 s for certificate + public DNS.")
        print("In-runtime check (goes out through the tailnet and back):")
        who = BASIC_USER if BASIC_PASS else "<the user you saved on Drive>"
        print(f"  curl -sS --socks5-hostname 127.0.0.1:1055 -u {who} "
              f"{URL}/system_stats")
        print()
        print("Funnel config:")
        print(tails("funnel", "status", check=False).stdout.strip() or "(no output)")
        print()
        print("tailscaled log: /content/tailscaled.log")

    # ---------------- Cloudflare quick tunnel (previous behaviour) ----------------
    elif TUNNEL_MODE == "cloudflare":
        CF = Path("/content/cloudflared")
        if not CF.exists():
            subprocess.run(
                ["bash", "-c",
                 "curl -L -o /content/cloudflared "
                 "https://github.com/cloudflare/cloudflared/releases/latest/download/"
                 "cloudflared-linux-amd64"],
                check=True,
            )
            CF.chmod(0o755)

        tunnel_cf_proc = subprocess.Popen(
            [str(CF), "tunnel", "--url", f"http://127.0.0.1:{UPSTREAM_PORT}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        TUNNEL_URL = None
        for _line in tunnel_cf_proc.stdout:
            _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", _line)
            if _m:
                TUNNEL_URL = _m.group(0)
                break
        print("Tunnel online:", TUNNEL_URL)
        print("Set these in the HF Space secrets and RESTART the Space:")
        print("  GATEWAY_UPSTREAM=" + str(TUNNEL_URL))
        print("  GATEWAY_BASIC_AUTH=" + BASIC_USER + ":<password you typed>")


## 12 · Diagnostics, cache release, and shutdown
Memory reported while idle includes cached model weights. Unload releases cache but the next job must reload. Only use these controls when no jobs are active, and pause outside clients first. Stopping ComfyUI does not delete the Colab VM; use Runtime → Disconnect and delete runtime when done.

In [ ]:
# @title Diagnostics / safe cleanup
UNLOAD_CACHED_MODELS=False # @param {type:"boolean"}
CLOSE_TUNNEL=False # @param {type:"boolean"}
STOP_COMFYUI=False # @param {type:"boolean"}
STOP_SUPERVISOR=False # @param {type:"boolean"}
# Any of these three must stop the supervisor first, or it restarts what you
# just shut down on the next beat.
if STOP_SUPERVISOR or STOP_COMFYUI or CLOSE_TUNNEL:
    if callable(globals().get("supervisor_stop")): supervisor_stop()
if CLOSE_TUNNEL:
    for name in ('tunnel_cf_proc','tunnel_caddy_proc','ts_proc'):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None: proc.terminate(); proc.wait(timeout=15)
    print('Tunnel closed.')
if UNLOAD_CACHED_MODELS and comfy_ready():
    require_idle()
    r=requests.post(BASE_URL+'/free',json={'unload_models':True,'free_memory':True},timeout=30);r.raise_for_status()
    time.sleep(5)
if STOP_COMFYUI:stop_owned_server()
gc.collect();resources()
if callable(globals().get('supervisor_status')): supervisor_status()
print('ComfyUI alive:',comfy_ready())
print('Last log lines:')
print(LOG.read_text(errors='replace')[-5000:] if LOG.exists() else 'No logs.')